# Predicting Stellar Class

Link to Competittion: https://www.kaggle.com/competitions/playground-series-s6e6/overview

## Imports

In [1]:
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['figure.figsize'] = (12, 6)

import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)

import xgboost as xgb
import optuna

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import auc, accuracy_score, confusion_matrix, mean_squared_error, classification_report, balanced_accuracy_score
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold, StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.cluster import KMeans
from sklearn.utils.class_weight import compute_sample_weight

from common import *

In [2]:
from platform import python_version
print('python: ', python_version())
print('pandas: ', pd.__version__)
print('numpy: ', np.__version__)
import matplotlib
print('matplotlib: ', matplotlib.__version__)
print('seaborn: ', sns.__version__)
import sklearn
print('sklearn: ', sklearn.__version__)
print('xgboost: ', xgb.__version__)

python:  3.13.13
pandas:  2.3.3
numpy:  2.4.6
matplotlib:  3.10.9
seaborn:  0.13.2
sklearn:  1.9.0
xgboost:  3.2.0


## Helpers

## Load data

In [16]:
train_df = pd.read_csv('archive/train.csv')
test_df = pd.read_csv('archive/test.csv')
orig_df = pd.read_csv('archive/star_classification.csv')

## Call the pipeline

In [17]:
df = (train_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

## Concat original dataset

In [18]:
orig_df_cleaned = (orig_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

In [19]:
# df = pd.concat([df, orig_df_cleaned])

## Features

In [20]:
target = get_target()

In [21]:
features = get_features(df)

In [22]:
features

['alpha',
 'delta',
 'u',
 'g',
 'r',
 'i',
 'z',
 'redshift',
 'spectral_type',
 'galaxy_population']

In [24]:
categorical_features = [
    'spectral_type',
    'galaxy_population'
]

In [25]:
numerical_features = [f for f in features if f not in categorical_features]

In [26]:
categorical_features

['spectral_type', 'galaxy_population']

In [27]:
numerical_features

['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']

In [28]:
df[features]

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence
...,...,...,...,...,...,...,...,...,...,...
577342,223.539288,2.503680,20.828729,18.854201,17.703108,17.190536,16.551356,0.511524,M,Red_Sequence
577343,223.895970,40.769343,23.734743,22.359173,20.697865,19.180264,18.947275,0.658589,M,Red_Sequence
577344,52.258927,0.671887,21.944250,21.215856,19.025966,18.772276,18.203397,0.376342,M,Red_Sequence
577345,247.362248,50.659819,21.969881,21.622766,20.987575,20.930924,21.478134,2.868359,G/K,Blue_Cloud


In [29]:
df[categorical_features] = df[categorical_features].astype('category')

In [30]:
categorical_features

['spectral_type', 'galaxy_population']

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 577347 entries, 0 to 577346
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype   
---  ------             --------------   -----   
 0   alpha              577347 non-null  float64 
 1   delta              577347 non-null  float64 
 2   u                  577347 non-null  float64 
 3   g                  577347 non-null  float64 
 4   r                  577347 non-null  float64 
 5   i                  577347 non-null  float64 
 6   z                  577347 non-null  float64 
 7   redshift           577347 non-null  float64 
 8   spectral_type      577347 non-null  category
 9   galaxy_population  577347 non-null  category
 10  _class             577347 non-null  int64   
dtypes: category(2), float64(8), int64(1)
memory usage: 40.7 MB


## Stratified KFold Loop

In [32]:
xgb_params = {
    'n_jobs': -1,
    'enable_categorical': True,
    'random_state': 123
}

In [33]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=123)
scores = []

X, y = df[features], df[target]

In [34]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = xgb.XGBClassifier(**xgb_params)
    fold_sample_weights = compute_sample_weight('balanced', y_train)
    model.fit(X_train, y_train, sample_weight=fold_sample_weights)

    preds = model.predict(X_val)
    score = balanced_accuracy_score(y_val, preds)
    scores.append(score)
    print(f"Fold {fold}:  {score:.5f}")

print(f"Mean: {np.mean(scores):.5f} ± {np.std(scores):.5f}")

Fold 0:  0.96202
Fold 1:  0.96164
Fold 2:  0.96426
Fold 3:  0.96425
Fold 4:  0.96306
Fold 5:  0.96456
Fold 6:  0.96173
Fold 7:  0.96450
Fold 8:  0.96380
Fold 9:  0.96509
Mean: 0.96349 ± 0.00122


## Optuna Hyperparameter Tuning

In [37]:
def objective(trial):
    params = {
        'n_jobs': -1,
        'enable_categorical': True,
        'random_state': 123,
        'n_estimators': 2000,
        'eval_metric': 'mlogloss',
        'early_stopping_rounds': 50,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 5.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        sw_tr = compute_sample_weight('balanced', y_tr)
        sw_val = compute_sample_weight('balanced', y_val)

        model = xgb.XGBClassifier(**params)
        model.fit(
            X_tr, y_tr,
            sample_weight=sw_tr,
            eval_set=[(X_val, y_val)],
            sample_weight_eval_set=[sw_val],
            verbose=False,
        )

        preds = model.predict(X_val)
        score = balanced_accuracy_score(y_val, preds)
        scores.append(score)

        trial.report(np.mean(scores), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)


# study = optuna.create_study(
#     direction='maximize',
#     pruner=optuna.pruners.MedianPruner(n_warmup_steps=2, n_startup_trials=5),
#     sampler=optuna.samplers.TPESampler(seed=123),
# )
# study.optimize(objective, n_trials=30, show_progress_bar=True)

# print(f"Best CV: {study.best_value:.5f}")
# print("Best params:")
# for k, v in study.best_params.items():
#     print(f"  {k}: {v}")

## Re-validate Best Params on 10-fold

In [38]:
best_params = {

}

In [39]:
best_params = {
    'n_jobs': -1,
    'enable_categorical': True,
    'random_state': 123,
    'n_estimators': 2000,
    'eval_metric': 'mlogloss',
    'early_stopping_rounds': 50,
    #**study.best_params,
    **best_params
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=123)
scores = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    sw_tr = compute_sample_weight('balanced', y_tr)
    sw_val = compute_sample_weight('balanced', y_val)

    model = xgb.XGBClassifier(**best_params)
    model.fit(X_tr, y_tr, sample_weight=sw_tr,
              eval_set=[(X_val, y_val)], sample_weight_eval_set=[sw_val], verbose=False)

    preds = model.predict(X_val)
    score = balanced_accuracy_score(y_val, preds)
    scores.append(score)
    print(f"Fold {fold}: {score:.5f}")

print(f"Mean: {np.mean(scores):.5f} ± {np.std(scores):.5f}")

Fold 0: 0.96368
Fold 1: 0.96347
Fold 2: 0.96567
Fold 3: 0.96600
Fold 4: 0.96328
Fold 5: 0.96544
Fold 6: 0.96281
Fold 7: 0.96549
Fold 8: 0.96605
Fold 9: 0.96629
Mean: 0.96482 ± 0.00127


## Final Model

In [40]:
final_params = {
    'n_jobs': -1,
    'enable_categorical': True,
    'random_state': 123,
    **best_params,
}
final_params.pop('early_stopping_rounds', None)
final_params.pop('eval_metric', None)

xgb_final_model = XGBClassifier(**final_params)
final_sample_weights = compute_sample_weight('balanced', y)
xgb_final_model.fit(X, y, sample_weight=final_sample_weights)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import lo

In [43]:
df = (test_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

In [45]:
df[categorical_features] = df[categorical_features].astype('category')

In [46]:
df

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence
...,...,...,...,...,...,...,...,...,...,...
247430,248.013903,49.533434,21.563545,21.716868,21.670791,21.265478,21.558618,1.214520,A/F,Blue_Cloud
247431,226.823885,52.635936,21.434441,21.075412,20.778300,20.962333,21.010822,1.004950,A/F,Blue_Cloud
247432,232.879335,44.948125,23.294670,22.336583,20.121142,19.405430,18.744581,0.269394,M,Red_Sequence
247433,351.396802,2.451824,20.882944,20.902010,20.222812,20.551737,20.278339,1.199392,G/K,Blue_Cloud


In [47]:
preds = xgb_final_model.predict(df[features])

In [91]:
preds

array([0, 0, 0, ..., 0, 1, 0], shape=(247435,))

In [92]:
submission_df = pd.DataFrame(test_df['id'].copy())

In [93]:
submission_df['class'] = preds

In [94]:
submission_df

,id,class
0,577347,0
1,577348,0
2,577349,0
3,577350,2
4,577351,0
...,...,...
247430,824777,1
247431,824778,1
247432,824779,0
247433,824780,1


In [95]:
submission_df['class'] = submission_df['class'].map({0: 'GALAXY', 1: 'QSO', 2: 'STAR'})

In [96]:
submission_df['class'].value_counts()

class
GALAXY    160449
QSO        50474
STAR       36512
Name: count, dtype: int64

In [110]:
try:
    last_submission = pd.read_csv(find_last_submission_file())
except:
    last_submission = None

In [111]:
submission_df

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
...,...,...
247430,824777,QSO
247431,824778,QSO
247432,824779,GALAXY
247433,824780,QSO


In [112]:
if last_submission == None:
    submission_df.to_csv('./archive/submission_01.csv', index=False)
    print('saving file')
else:
    
    if all(last_submission['class'].value_counts() == submission_df['class'].value_counts()):
        # they are the same, don't same
        print('skipping save')
    else:
        submission_df.to_csv(find_next_submission_file(), index=False)
        print('saving file')

saving file
